# Earthquake deaths, magnitude and income: NOAA Data Collection

**Source:** NOAA NCEI Significant Earthquake Database
**Goal:** Collect historical fatality counts, damage estimates, and location metadata for significant earthquakes globally from 2000 to 2025.

This notebook executes the API collection end-to-end, documents `robots.txt` and Terms of Use, and saves the raw JSON file untouched into the `raw/` directory.

In [5]:
import os
import json
import time
import requests
import pandas as pd
from datetime import date
from urllib.robotparser import RobotFileParser

# 1. Setup required directories
os.makedirs("raw", exist_ok=True)
USER_AGENT = "DA2402_Project_Team/1.0 (contact: student@smail.iitm.ac.in)"

## 1. Check robots.txt and Terms of Use
We verify that our scraper is allowed to access the NOAA API and acknowledge their crawl delay. NOAA data is a U.S. Government work in the Public Domain.

In [6]:
robots_url = "https://www.ngdc.noaa.gov/robots.txt"
api_url = "https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/earthquakes"

resp_robots = requests.get(robots_url, headers={"User-Agent": USER_AGENT})
print(f"Robots.txt Status: {resp_robots.status_code}")
print("--- First 15 lines of robots.txt for the report ---")
print("\n".join(resp_robots.text.splitlines()[:15]))

# Parse rules to prove compliance
parser = RobotFileParser()
parser.parse(resp_robots.text.splitlines())
is_allowed = parser.can_fetch(USER_AGENT, api_url)

print(f"\nAm I allowed to scrape the API? {is_allowed}")
print("Terms of Use: NOAA NCEI data is Public Domain (U.S. Government Work).")
print("Note: robots.txt specifies a Crawl-delay of 60. We implement a local cache below to ensure we do not repeatedly hit the server while re-running the notebook.")

Robots.txt Status: 200
--- First 15 lines of robots.txt for the report ---
User-agent: *
Crawl-delay: 60
Disallow: /catalog-index
Disallow: /cgi-bin
Disallow: /dmsp/cgi-bin
Disallow: /dmsp/data
Disallow: /dmsp/include
Disallow: /dmsp/protected
Disallow: /eog
Disallow: /geomag/cdroms
Disallow: /geomag/data
Disallow: /geomag/EMM/data
Disallow: /geomag/pmag/datafiles
Disallow: /geomag/WMM/data
Disallow: /globe

Am I allowed to scrape the API? True
Terms of Use: NOAA NCEI data is Public Domain (U.S. Government Work).
Note: robots.txt specifies a Crawl-delay of 60. We implement a local cache below to ensure we do not repeatedly hit the server while re-running the notebook.


## 2. End-to-End API Collection
The NOAA API uses pagination. We use a `while` loop to fetch pages sequentially until we hit the `totalPages` limit. To respect the crawl-delay, the script will skip the download if the raw file already exists locally.

In [7]:
raw_filepath = f"raw/noaa_sig_eq_2000_2025.json"

if os.path.exists(raw_filepath):
    print(f"Raw file {raw_filepath} already exists. Loaded from cache to respect crawl-delay.")
else:
    print(f"Fetching NOAA data from: {api_url}")
    all_items = []
    page = 1
    
    while True:
        params = {"minYear": 2000, "maxYear": 2025, "page": page}
        r = requests.get(api_url, params=params, headers={"User-Agent": USER_AGENT})
        r.raise_for_status()
        
        data = r.json()
        items = data.get("items", [])
        if not items:
            break
            
        all_items.extend(items)
        total_pages = data.get("totalPages", 1)
        print(f" Fetched page {page} of {total_pages} ({len(items)} records)")
        
        if page >= total_pages:
            break
            
        page += 1
        time.sleep(1) # Polite gap between pagination requests

    # Save exactly as it came from the API
    with open(raw_filepath, "w", encoding="utf-8") as f:
        json.dump({
            "source_url": api_url, 
            "items": all_items
        }, f, indent=2)
        
    print(f"\nSuccess! Saved {len(all_items)} raw records to {raw_filepath}")

Fetching NOAA data from: https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/earthquakes
 Fetched page 1 of 8 (200 records)
 Fetched page 2 of 8 (200 records)
 Fetched page 3 of 8 (200 records)
 Fetched page 4 of 8 (200 records)
 Fetched page 5 of 8 (200 records)
 Fetched page 6 of 8 (200 records)
 Fetched page 7 of 8 (200 records)
 Fetched page 8 of 8 (54 records)

Success! Saved 1454 raw records to raw/noaa_sig_eq_2000_2025.json


## 3. Raw Data Inspection
Loading the raw JSON into a pandas DataFrame to display the shape, `dtypes`, and state the meaning of every kept column before moving to the cleaning phase.

In [8]:
# Load the raw file
with open(raw_filepath, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df_raw = pd.DataFrame(raw_data["items"])

# We define the columns we actually need for the analysis
kept_cols = [
    "id", "year", "month", "day", "country", "locationName",
    "latitude", "longitude", "eqDepth", "eqMagnitude",
    "deaths", "deathsAmountOrder", "damageMillionsDollars"
]

# Ensure we only select columns that exist in the raw response
kept_cols = [c for c in kept_cols if c in df_raw.columns]
df_kept = df_raw[kept_cols]

print("--- Raw Data Inspection ---")
print(f"Shape of kept data: {df_kept.shape}")
print(f"\nDtypes:\n{df_kept.dtypes}")

# Explicitly state the meaning of every kept column for the rubric
column_meanings = {
    "id": "Unique integer ID for the event in NOAA's database",
    "year": "Event year",
    "month": "Event month",
    "day": "Event day",
    "country": "Country name (all-caps, will need standardization)",
    "locationName": "Text description of the affected region",
    "latitude": "Latitude coordinate",
    "longitude": "Longitude coordinate",
    "eqDepth": "Focal depth in km",
    "eqMagnitude": "Reported earthquake magnitude",
    "deaths": "Total number of deaths reported",
    "deathsAmountOrder": "Categorical scale (1 to 4) representing range of deaths if exact count is unknown",
    "damageMillionsDollars": "Direct economic damage in millions of USD"
}

print("\nMeaning of kept columns:")
for col in kept_cols:
    print(f" - {col}: {column_meanings[col]}")

--- Raw Data Inspection ---
Shape of kept data: (1454, 13)

Dtypes:
id                         int64
year                       int64
month                      int64
day                        int64
country                      str
locationName                 str
latitude                 float64
longitude                float64
eqDepth                  float64
eqMagnitude              float64
deaths                   float64
deathsAmountOrder        float64
damageMillionsDollars    float64
dtype: object

Meaning of kept columns:
 - id: Unique integer ID for the event in NOAA's database
 - year: Event year
 - month: Event month
 - day: Event day
 - country: Country name (all-caps, will need standardization)
 - locationName: Text description of the affected region
 - latitude: Latitude coordinate
 - longitude: Longitude coordinate
 - eqDepth: Focal depth in km
 - eqMagnitude: Reported earthquake magnitude
 - deaths: Total number of deaths reported
 - deathsAmountOrder: Categorical scal